# Bulk Sharing Automation — Tag-Based Group & Access Level Management

Automates sharing of AGOL items to groups and sets access levels (`org`, `public`, `group`, `private`) based on configurable tag rules. Includes dry-run preview, validation, and detailed logging.

## How to use

1. Run the **Connection** cell below.
2. Edit the `RULES` list in the **Configuration** cell. Each rule specifies:
   - `tags`: list of required tags (all must match, case-insensitive)
   - `access`: *(optional)* one of `org`, `public`, `group`, or `private`
   - `groups`: list of group titles or IDs to share matching items to
3. Set `DRY_RUN = True` and run all cells to **preview** what would change.
4. Review the summary output. When satisfied, set `DRY_RUN = False` and run again to **apply**.

## Requirements

- The running account must be a **member** of all target groups.
- Items must be **owned** by the running account to be shared.

---
**Tags:** automation, sharing, groups, content management, notebook, geospatial hub

> **Terms of Use:** Review and adapt the `RULES` configuration to your own content and groups before running.

## Cell 1 — Connection

In [ ]:
from arcgis.gis import GIS

# Connect using the active portal profile (works inside ArcGIS Notebooks).
# Replace 'home' with your own GIS URL / credentials if running outside a notebook server.
gis = GIS('home')
print(f"Connected as: {gis.users.me.username}")

## Cell 2 — Configuration

Edit `RULES` and `DRY_RUN` here before running the notebook.

In [ ]:
import logging
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
LOG_FILE = Path('/arcgis/home/agol_sharing.log')

logger = logging.getLogger('agol_sharing')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()  # avoid duplicate handlers on re-runs

_fmt = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s')

_fh = logging.FileHandler(LOG_FILE, encoding='utf-8')
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

_ch = logging.StreamHandler(sys.stdout)
_ch.setFormatter(_fmt)
logger.addHandler(_ch)

# ---------------------------------------------------------------------------
# Rules
# ---------------------------------------------------------------------------
# Each rule is a dict with:
#   tags   – list[str]  All tags an item must have to match (case-insensitive).
#   access – str | None  Target access level: 'org', 'public', 'group', or 'private'.
#            Omit (or set None) to leave the current access level unchanged.
#   groups – list[str]  Group titles or IDs to share matching items to.
#            May be an empty list if you only want to change the access level.

RULES = [
    {
        'tags':   ['overture', 'public-release'],
        'access': 'public',
        'groups': ['Overture Maps Public'],
    },
    {
        'tags':   ['overture', 'internal'],
        'access': 'org',
        'groups': ['Overture Maps Internal', 'Geospatial Hub'],
    },
    {
        'tags':   ['archive'],
        'access': 'private',
        'groups': [],
    },
]

# ---------------------------------------------------------------------------
# Dry-run flag
# ---------------------------------------------------------------------------
# Set to False to apply changes.
DRY_RUN: bool = True

# Maximum number of items to fetch per query.
MAX_ITEMS_PER_QUERY: int = 10000

VALID_ACCESS_LEVELS = {'org', 'public', 'group', 'private'}

logger.info("Configuration loaded. DRY_RUN=%s, rules=%d", DRY_RUN, len(RULES))

## Cell 3 — Pre-flight Validation

Validates the configuration and resolves group titles to objects before any items are touched.

In [ ]:
def validate_config(rules, gis_conn):
    """Validate RULES and resolve group names/IDs to Group objects.

    Returns a list of resolved rules where each 'groups' value is a list of
    arcgis.gis.Group objects.

    Raises ValueError if any rule is malformed or a group cannot be found.
    """
    if not rules:
        raise ValueError("RULES list is empty — nothing to do.")

    resolved_rules = []
    errors = []

    for idx, rule in enumerate(rules):
        rule_label = f"Rule[{idx}]"

        # --- tags ---------------------------------------------------------
        tags = rule.get('tags')
        if not tags or not isinstance(tags, list):
            errors.append(f"{rule_label}: 'tags' must be a non-empty list.")
            continue

        # --- access -------------------------------------------------------
        access = rule.get('access')
        if access is not None and access not in VALID_ACCESS_LEVELS:
            errors.append(
                f"{rule_label}: 'access' value '{access}' is invalid. "
                f"Choose from {sorted(VALID_ACCESS_LEVELS)}."
            )
            continue

        # --- groups -------------------------------------------------------
        group_refs = rule.get('groups', [])
        if not isinstance(group_refs, list):
            errors.append(f"{rule_label}: 'groups' must be a list.")
            continue

        resolved_groups = []
        for ref in group_refs:
            # Try by ID first, then by title.
            group = None
            try:
                group = gis_conn.groups.get(ref)
            except Exception:
                pass

            if group is None:
                results = gis_conn.groups.search(f'title:"{ref}"', max_groups=5)
                exact = [g for g in results if g.title.lower() == ref.lower()]
                if exact:
                    group = exact[0]

            if group is None:
                errors.append(
                    f"{rule_label}: Group '{ref}' not found or you are not a member."
                )
            else:
                resolved_groups.append(group)
                logger.debug("%s: resolved group '%s' → id=%s", rule_label, ref, group.id)

        if not errors or all(e.startswith(f"Rule[{idx}]") and 'Group' in e for e in errors):
            resolved_rules.append({
                'tags': [t.lower() for t in tags],
                'access': access,
                'groups': resolved_groups,
                '_label': rule_label,
            })

    if errors:
        for err in errors:
            logger.error("Validation error — %s", err)
        raise ValueError("Configuration validation failed:\n" + "\n".join(errors))

    return resolved_rules


logger.info("Running pre-flight validation...")
try:
    resolved_rules = validate_config(RULES, gis)
    logger.info("Validation passed. %d rule(s) ready.", len(resolved_rules))
    print(f"✅  Validation passed — {len(resolved_rules)} rule(s) ready.")
except ValueError as exc:
    logger.error("Aborting due to validation errors.")
    raise

## Cell 4 — Main Execution

Fetches all owned items, matches them against rules, then shares/updates access levels.

In [ ]:
from collections import defaultdict


def item_matches_rule(item, rule):
    """Return True if all rule tags are present on the item (case-insensitive)."""
    item_tags_lower = {t.lower() for t in (item.tags or [])}
    return all(rt in item_tags_lower for rt in rule['tags'])


def apply_rule(item, rule, dry_run):
    """Apply a single rule to an item. Returns a dict summarising what happened."""
    result = {
        'item_id': item.id,
        'item_title': item.title,
        'rule': rule['_label'],
        'access_change': None,
        'groups_shared': [],
        'errors': [],
    }

    # --- Access level -----------------------------------------------------
    target_access = rule.get('access')
    if target_access and item.access != target_access:
        if dry_run:
            result['access_change'] = f"{item.access} → {target_access} (DRY RUN)"
            logger.info(
                "[DRY RUN] Would set access '%s' → '%s' for item '%s' (%s)",
                item.access, target_access, item.title, item.id,
            )
        else:
            try:
                item.share(everyone=(target_access == 'public'),
                           org=(target_access == 'org'))
                if target_access == 'private':
                    item.unshare(everyone=True, org=True)
                result['access_change'] = f"{item.access} → {target_access}"
                logger.info(
                    "Set access '%s' for item '%s' (%s)",
                    target_access, item.title, item.id,
                )
            except Exception as exc:
                msg = f"Failed to set access '{target_access}': {exc}"
                result['errors'].append(msg)
                logger.error("%s — item '%s' (%s)", msg, item.title, item.id)
    elif target_access:
        result['access_change'] = f"already {target_access} (no change)"

    # --- Group sharing ----------------------------------------------------
    for group in rule.get('groups', []):
        if dry_run:
            result['groups_shared'].append(f"{group.title} (DRY RUN)")
            logger.info(
                "[DRY RUN] Would share item '%s' (%s) to group '%s'",
                item.title, item.id, group.title,
            )
        else:
            try:
                item.share(groups=[group])
                result['groups_shared'].append(group.title)
                logger.info(
                    "Shared item '%s' (%s) to group '%s'",
                    item.title, item.id, group.title,
                )
            except Exception as exc:
                msg = f"Failed to share to group '{group.title}': {exc}"
                result['errors'].append(msg)
                logger.error("%s — item '%s' (%s)", msg, item.title, item.id)

    return result


# ---------------------------------------------------------------------------
# Fetch items owned by the connected user
# ---------------------------------------------------------------------------
me = gis.users.me
logger.info("Fetching items for user '%s'...", me.username)
owned_items = me.items(max_items=MAX_ITEMS_PER_QUERY)

# Also check items inside root folders
for folder in me.folders:
    owned_items.extend(me.items(folder=folder, max_items=MAX_ITEMS_PER_QUERY))

logger.info("Found %d owned item(s).", len(owned_items))

# ---------------------------------------------------------------------------
# Match items against rules and apply
# ---------------------------------------------------------------------------
all_results = []
matched_counts = defaultdict(int)

for item in owned_items:
    for rule in resolved_rules:
        if item_matches_rule(item, rule):
            matched_counts[rule['_label']] += 1
            result = apply_rule(item, rule, DRY_RUN)
            all_results.append(result)

# ---------------------------------------------------------------------------
# Summary output
# ---------------------------------------------------------------------------
mode_label = '🔍 DRY RUN — no changes applied' if DRY_RUN else '✅ Changes applied'
print(f"\n{'='*60}")
print(f"  {mode_label}")
print(f"  Total items processed : {len(owned_items)}")
print(f"  Total rule matches    : {len(all_results)}")
print(f"{'='*60}")

for rule in resolved_rules:
    label = rule['_label']
    print(f"\n  {label} (tags={rule['tags']}, access={rule['access']})")
    print(f"    Matched items : {matched_counts[label]}")

print(f"\n{'='*60}")
print("  Itemised results:")
for r in all_results:
    print(f"\n  [{r['rule']}] '{r['item_title']}' ({r['item_id']})")
    if r['access_change']:
        print(f"    Access  : {r['access_change']}")
    for g in r['groups_shared']:
        print(f"    Group   : {g}")
    for e in r['errors']:
        print(f"    ⚠️  ERROR: {e}")

logger.info(
    "Execution complete. DRY_RUN=%s, items_processed=%d, rule_matches=%d",
    DRY_RUN, len(owned_items), len(all_results),
)

## Cell 5 — Post-run Validation

Confirms that items are actually shared to the expected groups after a live run.
Skipped automatically in dry-run mode.

In [ ]:
if DRY_RUN:
    print("ℹ️  Post-run validation skipped in DRY_RUN mode.")
    logger.info("Post-run validation skipped (DRY_RUN=True).")
else:
    logger.info("Running post-run validation...")
    validation_failures = []

    for r in all_results:
        if r['errors']:
            # Skip items that already had errors during apply.
            continue

        item = gis.content.get(r['item_id'])
        if item is None:
            validation_failures.append(f"Item {r['item_id']} not found during validation.")
            continue

        # Check access level
        for rule in resolved_rules:
            if rule['_label'] != r['rule']:
                continue
            target_access = rule.get('access')
            if target_access and item.access != target_access:
                msg = (
                    f"Item '{item.title}' ({item.id}): "
                    f"expected access '{target_access}', got '{item.access}'."
                )
                validation_failures.append(msg)
                logger.warning("Validation FAIL — %s", msg)

            # Check group membership
            item_group_ids = {g['id'] for g in (item.shared_with.get('groups') or [])}
            for group in rule.get('groups', []):
                if group.id not in item_group_ids:
                    msg = (
                        f"Item '{item.title}' ({item.id}): "
                        f"not found in group '{group.title}' ({group.id})."
                    )
                    validation_failures.append(msg)
                    logger.warning("Validation FAIL — %s", msg)

    if validation_failures:
        print(f"\n⚠️  Post-run validation: {len(validation_failures)} issue(s) found:")
        for f in validation_failures:
            print(f"   - {f}")
        logger.warning("Post-run validation finished with %d issue(s).", len(validation_failures))
    else:
        print("✅  Post-run validation passed — all items confirmed in expected groups/access.")
        logger.info("Post-run validation passed.")

print(f"\nLog file: {LOG_FILE}")